# Task: Car Price Prediction

## Import libraries


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
# pytorch
import torch
import torch.nn.functional as torch_functional  # Parameterless functions, like (some) activation functions
from torch.utils.data import DataLoader, Dataset
from torch import optim  # For optimizers like SGD, Adam, etc.
from torch import nn  # All neural network modules
from torch.utils.data import (
    DataLoader,
)  # Gives easier dataset managment by creating mini batches etc.

## Data Set Information:

This data set consists of three types of entities: (a) the specification of an auto in terms of various characteristics, (b) its assigned insurance risk rating, (c) its normalized losses in use as compared to other cars. The second rating corresponds to the degree to which the auto is more risky than its price indicates. Cars are initially assigned a risk factor symbol associated with its price. Then, if it is more risky (or less), this symbol is adjusted by moving it up (or down) the scale. Actuarians call this process "symboling". A value of +3 indicates that the auto is risky, -3 that it is probably pretty safe.

The third factor is the relative average loss payment per insured vehicle year. This value is normalized for all autos within a particular size classification (two-door small, station wagons, sports/speciality, etc...), and represents the average loss per car per year.

Note: Several of the attributes in the database could be used as a "class" attribute.

## Data Dictionary

1. symboling: -3, -2, -1, 0, 1, 2, 3.
2. normalized-losses: continuous from 65 to 256.
3. make:
alfa-romero, audi, bmw, chevrolet, dodge, honda,
isuzu, jaguar, mazda, mercedes-benz, mercury,
mitsubishi, nissan, peugot, plymouth, porsche,
renault, saab, subaru, toyota, volkswagen, volvo

4. fuel-type: diesel, gas.
5. aspiration: std, turbo.
6. num-of-doors: four, two.
7. body-style: hardtop, wagon, sedan, hatchback, convertible.
8. drive-wheels: 4wd, fwd, rwd.
9. engine-location: front, rear.
10. wheel-base: continuous from 86.6 120.9.
11. length: continuous from 141.1 to 208.1.
12. width: continuous from 60.3 to 72.3.
13. height: continuous from 47.8 to 59.8.
14. curb-weight: continuous from 1488 to 4066.
15. engine-type: dohc, dohcv, l, ohc, ohcf, ohcv, rotor.
16. num-of-cylinders: eight, five, four, six, three, twelve, two.
17. engine-size: continuous from 61 to 326.
18. fuel-system: 1bbl, 2bbl, 4bbl, idi, mfi, mpfi, spdi, spfi.
19. bore: continuous from 2.54 to 3.94.
20. stroke: continuous from 2.07 to 4.17.
21. compression-ratio: continuous from 7 to 23.
22. horsepower: continuous from 48 to 288.
23. peak-rpm: continuous from 4150 to 6600.
24. city-mpg: continuous from 13 to 49.
25. highway-mpg: continuous from 16 to 54.
26. price: continuous from 5118 to 45400.		

**Source: https://archive.ics.uci.edu/ml/datasets/Automobile


## Read data

In [ ]:
df = pd.read_csv('CarPrice.csv')

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

## Data Preprocessing

In [ ]:
df['CarName'].nunique()

## Label encoder

In [ ]:
from sklearn.preprocessing import LabelEncoder

### encode the categorical features

In [ ]:
cat_features = [feature for feature in df.columns if df[feature].dtype == 'object']
cat_features

In [ ]:
encoder = LabelEncoder()

for feature in cat_features:
    df[feature] = encoder.fit_transform(df[feature])

In [ ]:
df['CarName'].nunique()

## Get the x and y data

In [ ]:
x = df.iloc[:, 1:-1]
x = x.drop('CarName', axis = 1)
y = pd.DataFrame(df['price'])

In [ ]:
x.head()

## Scaling

### Standard scaling

In [ ]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()

In [ ]:
x_scaled = sc.fit_transform(x)

In [ ]:
pd.DataFrame(x_scaled)

## Splitting the dataset into the Training set and Test set

In [ ]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x_scaled, y, test_size = 0.2, random_state = 0)

## size/shape of dataframe

In [ ]:
n_samples = x_train.shape[0]
n_featuers = x_train.shape[1]
print(f'n_samples: {n_samples}, n_features: {n_featuers}')

## Set device

## Implement FCNN

In [ ]:
class NN(nn.Module):
    def __init__(self, input_size):

        super(NN, self).__init__()
        self.fully_connected_1 = nn.Linear(input_size, 128)
        self.fully_connected_2 = nn.Linear(128, 64)
        self.fully_connected_3 = nn.Linear(64, 4)
        self.fully_connected_4 = nn.Linear(4, 1)
        self.dropout = nn.Dropout(p=0.5)

    def forward(self, x):

        x = torch_functional.softmax(self.fully_connected_1(x))
        x = self.dropout(x) 
        x = torch_functional.softmax(self.fully_connected_2(x))
        x = torch_functional.softmax(self.fully_connected_3(x))
        x = torch_functional.softmax(self.fully_connected_4(x))
        return x

## Hyperparameters

In [ ]:
input_size = n_featuers
learning_rate = 0.01
batch_size = 10
num_epochs = 50

## Load Data

In [ ]:
class CustomDataset(Dataset):

    def __init__(self, x_data, y_data):
        
        self.inp_data = torch.FloatTensor(x_data)
        self.out_data = torch.FloatTensor(y_data.values)
        
    def __len__(self):
        return len(self.inp_data)
    
    def __getitem__(self, index):
 
        return self.inp_data[index],self.out_data[index]

In [ ]:
# training and validation dataset 
train_dataset = CustomDataset(x_train, y_train)
test_dataset = CustomDataset(x_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True)

In [ ]:
train_dataset[0]

In [ ]:
test_dataset[0]

## Initialize network

In [ ]:
# Set the device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Move the model to the device

model = NN(input_size=input_size).to(device)

## Loss and optimizer

In [ ]:
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

## Train Network

In [ ]:
def train_one_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    total_batch_loss = 0
    total_batches = 0

    for x, y in train_loader:
        # Move x to the device
        x, y = x.to(device), y.to(device)

        # Forward pass
        prediction = model(x)
        loss = torch.sqrt(criterion(prediction, y)) 

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Accumulate loss
        total_batch_loss += loss.item()
        total_batches += 1

    avg_loss = total_batch_loss / total_batches
    return avg_loss



In [ ]:

def evaluate(model, test_loader, criterion, device):
    model.eval()
    total_batch_loss = 0
    total_batches = 0

    with torch.no_grad():
        for x, y in test_loader:
            # Move x to the device
            x, y = x.to(device), y.to(device)

            # Forward pass
            test_pred = model(x)
            test_loss = criterion(test_pred, y)  # MSE loss

            # Accumulate loss
            total_batch_loss += test_loss.item()
            total_batches += 1

    avg_loss = total_batch_loss / total_batches
    return avg_loss



In [ ]:

# Main training loop
loss_epoch = []
test_loss_epoch = []

for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    test_loss = evaluate(model, test_loader, criterion, device)

    loss_epoch.append(train_loss)
    test_loss_epoch.append(test_loss)

    if epoch % 10 == 0:
        print(f"Epoch: {epoch + 1}/{num_epochs}, Train Loss: {train_loss:.4f}, Test Loss: {test_loss:.4f}")


## Train loss per epoch

In [ ]:
plt.plot(loss_epoch)
plt.plot(test_loss_epoch, color = 'orange')

## MSE

In [ ]:
def check_mse(loader, model):
    mse = 0
    num_samples = 0
    model.eval()

    # We don't need to keep track of gradients here so we wrap it in torch.no_grad()
    with torch.no_grad():
        # Loop through the data
        for i, (x, y) in enumerate(loader):
#             print(i, x.shape, y.shape)
            # Move data to device
            x = x.to(device=device)
            y = y.to(device=device)

            # Get to correct shape
            x = x.reshape(x.shape[0], -1)

            # Forward pass
            prediction = model(x)

            # compute the loss (mean squared error)
            mse += torch.sum((y - prediction)**2)
            
            # Keep track of number of samples
            num_samples += prediction.size(0)
#             print(mse, num_samples)
    model.train()
    return mse

## Model performance

In [ ]:
# Check accuracy on training & test to see how good our model
print(f"Accuracy on training set: {check_mse(train_loader, model)*100:.2f}")
print(f"Accuracy on test set: {check_mse(test_loader, model)*100:.2f}")